In [1]:
import pandas as pd, numpy as np, os, warnings,sys,math
from collections import defaultdict

In [2]:
yr = 2024

In [3]:
warnings.filterwarnings('ignore')

In [4]:
source_dir = [f'../../原始数据集/{yr}/goldenbull_zqzg/upload_data',f'../../原始数据集/{yr}/goldenbull_zqzg/upload_data_ex']
user_info = pd.read_excel(f'../../原始数据集/{yr}/用户信息表.xlsx',skiprows=[0],dtype={'ID':int})

In [5]:
user_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      88 non-null     int32 
 1   用户名     88 non-null     object
 2   电话      88 non-null     object
 3   邮箱      88 non-null     object
 4   公司代码    88 non-null     object
 5   公司地址    35 non-null     object
 6   公司名称    60 non-null     object
 7   公司备案    13 non-null     object
 8   注册地址    35 non-null     object
 9   办公地址    60 non-null     object
 10  注册时间    60 non-null     object
 11  公司简介    37 non-null     object
dtypes: int32(1), object(11)
memory usage: 8.0+ KB


In [6]:
def print_progress(current, total, bar_length=50,extra_text=''):
    progress = current / total
    block = int(round(bar_length * progress))
    text = f"[{'#' * block + '-' * (bar_length - block)}] {int(progress * 100)}% | ({current}/{total}):{extra_text}"

    # 使用 '\r' 返回行首，'\033[K' 清除行内剩余字符
    sys.stdout.write(f'\r{text}\033[K')
    sys.stdout.flush()


In [21]:
num_dict = {29:'长江资管',46:'广发资管',47:'太平洋证券',53:'天风资管',55:'第一创业',58:'万联资管',64:'东兴资管',70:'方正资管'}

In [31]:
sheetname = ['基础信息','产品净值','历任基金经理信息','规模','拆分','分红']

company_dict= {}
error_file = []

for src in source_dir:
    for subdir, dirs, files in os.walk(src):
        folder_name = os.path.basename(subdir)
        if folder_name in ['1','null']:
            continue
        else:
            try:
                num = int(folder_name)
            except:
                continue
                
            if num not in company_dict.keys():
                company_dict[num] = defaultdict(list)
            if '公司名称' not in company_dict[num].keys():
                company_name_org = str(user_info.loc[user_info['ID'] == num,'公司名称'].values[0])
                if isinstance(company_name_org,str) and company_name_org != 'nan':
                    company_name = company_name_org[:4] if '太平洋' not in company_name_org else company_name_org[:5]
                else:
                    company_name = num_dict[num]
            else:
                company_name = company_dict[num]['公司名称']
            print('\n\n')
            print(f'{num} {company_name}'.center(80,'-'))
            i = 1
            er_file = []
        for file in files: 
            # 检查文件名中是否包含'.xls'
            if '.xls' in file.lower():
                file_path = os.path.join(subdir, file)
                xls = pd.ExcelFile(file_path)
                sn = xls.sheet_names
                if len(sn) < 4:
                    print(sn)
                    error_file.append(file)
                    er_file.append(file)
                    continue
                else:
                    for sheet_name in sn:
                        df = pd.read_excel(xls,sheet_name =sheet_name)
                        
                        # 规范化
                        if '基础' in sheet_name: sheet_name = '基础信息'
                        elif '净值' in sheet_name: sheet_name = '产品净值'
                        elif '经理' in sheet_name: sheet_name = '历任基金经理信息'
                        elif '分红' in sheet_name: sheet_name = '分红'
                        elif '规模' in sheet_name: sheet_name = '规模'
                        elif '拆分' in sheet_name: sheet_name = '拆分'
                         
                        if sheet_name == '基础信息':
                            cols = df.columns.to_list()
                            if '公司名称' not in company_dict[num].keys():
                                #comfirm = input(f'\n输入规定的{num}公司名称,现<<{company_name}>>原<<{company_name_org}>>\n(若已可行，回车即可)：')
                                confirm = ''
                                """注：新一年的评奖时把封印解开！"""
                                if not comfirm:
                                    pass
                                else:
                                    company_name = comfirm
                                company_dict[num]['公司名称'] = company_name
                            else:
                                company_name = company_dict[num]['公司名称']
                            df['公司名称'] = company_name
                            df = df[['公司名称']+ cols]

                        cols = df.columns.to_list()
                        col = [elm.strip('*').strip() for elm in cols]
                        df.columns = col
                        company_dict[num][sheet_name].append(df)
                    
                    if i + 1 <= len(files):
                        i = i + 1
                    print_progress(i,len(files),10,f'正在读取:{file}')
                if '分红' not in company_dict[num].keys():
                        print(f'\n{file}少了分红')
                        company_dict[num]['分红'].append(pd.DataFrame(data=None,columns=['产品全称','分红日期','每份分红金额(元)']))
                if '拆分' not in company_dict[num].keys():
                    print(f'\n{file}少了拆分')
                    company_dict[num]['拆分'].append(pd.DataFrame(data=None,columns=['产品全称','拆分日期','拆分比例']))
            else:
                error_file.append(file)
                er_file.append(file)
        if len(er_file)>0:
            print(f"\n出错文件：{','.join(er_file)}")
                        
                    
        

    




------------------------------------11 中信期货-------------------------------------
[##########] 100% | (4/4):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表.xlsx-87306454
出错文件：券商集合资产管理人评选指标补充.docx-02426627



------------------------------------13 华宝证券-------------------------------------
[##########] 100% | (3/3):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-华宝证券资管.xlsx-5157171015255


------------------------------------14 华创证券-------------------------------------
[##########] 100% | (2/2):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-华创证券.xlsx-44201797


------------------------------------15 国金证券-------------------------------------
[##########] 100% | (1/1):正在读取:国金证券资产管理有限公司2024证券公司集合资管计划金牛奖机构填报数据表-.xlsx-74538611


------------------------------------16 安信证券-------------------------------------
[##########] 100% | (2/2):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表.xlsx-2262542933


------------------------------------18 东莞证券-------------------------------------
[##########] 100% | (3/3):正在读取:小集合资产管理计划产品奖项（年度-券商集合资产管理计划）.xlsx-44722995K


------

[###-------] 33% | (4/12):正在读取:【国海证券】2024证券公司集合资产管理计划金牛奖机构填报数据表.xlsx-83844336['Sheet1', 'Sheet2', 'Sheet3']
[####------] 41% | (5/12):正在读取:【投资策略已更新】【国海证券】2024证券公司集合资产管理计划金牛奖机构填报数据表.xlsx-10946141['Sheet1', 'Sheet2', 'Sheet3']
['Sheet1']

出错文件：【产品规模】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-77829701,【净值、机构占比等】2024证券公司集合资产管理计划金牛奖机构填报数据表(1).xlsx-57615308,【净值表】2024证券公司集合资产管理计划金牛奖机构填报数据表.xlsx-38954870,【净值表】2024证券公司集合资产管理计划金牛奖机构填报数据表.xlsx-53786567,【分红记录】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-13701780,【基金经理信息】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-17006055,【拆分记录】2024证券公司集合资产管理计划金牛奖机构填报数据表（国海证券）.xlsx-63102060,【机构占比已更新】【净值表】2024证券公司集合资产管理计划金牛奖机构填报数据表.xlsx-19792992



------------------------------------68 中国国际-------------------------------------
[##########] 100% | (3/3):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-中金公司.xlsx-65154672


------------------------------------69 江海证券-------------------------------------
[##########] 100% | (2/2):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表（江海证券）.xlsx-96779888K


----------------------

[##########] 100% | (1/1):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-华鑫证券-修正.xlsx-09805245


------------------------------------52 财通证券-------------------------------------

出错文件：固收财通证券资管-2024资管金牛奖填报.pdf-35973712,权益财通证券资管-2024资管金牛奖填报.pdf-61508239,财通证券资管2024资管金牛申报材料.docx-22223143,财通证券资管2024资管金牛申报材料.pdf-10237595



------------------------------------53 天风资管-------------------------------------

出错文件：盖章附件（用印版）.pdf-96087035



------------------------------------57 兴证证券-------------------------------------

出错文件：2024证券公司集合资管计划金牛奖机构填报数据表.pdf-02699646



------------------------------------58 万联资管-------------------------------------

出错文件：万联添利粤升1号.png-82685808,万联聚享添利1号.png-43189603,万联聚享添利3号.png-72090783,万联聚享添利5号.png-53763315,万联证券添利粤信3号.png-34756769,万联证券资产管理（广东）有限公司【万联资管】集合产品WIND同类产品排名2023年12月末截图证明.docx-68411032,万联证券资产管理（广东）有限公司【万联资管】集合产品WIND同类产品排名2023年12月末截图证明——PART1.docx-52022104,万联证券资产管理（广东）有限公司【万联资管】集合产品WIND同类产品排名2023年12月末截图证明——PART2.docx-49746312,万联财星东照月添利1号.png-12624428,产品基本信息——万联资管.pdf-03890073,

In [32]:
company_dict.keys()

dict_keys([11, 13, 14, 15, 16, 18, 19, 20, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40, 42, 44, 45, 46, 47, 48, 49, 50, 52, 53, 55, 57, 58, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 78, 79, 83, 84, 86, 87, 88, 89, 91, 92, 93, 94])

In [33]:
for i in company_dict.keys():
    print(company_dict[i]['公司名称'],end=' ')
number = len(company_dict.keys())
print()
print(f'总共有{number}家公司参评')

中信期货 华宝证券 华创证券 国金证券 安信证券 东莞证券 中航证券 野村东方 西部证券 德邦证券 申万宏源 浙江浙商 渤海汇金 中信证券 国元证券 长江资管 上海光大 首创证券 中泰证券 诚通证券 西南证券 华泰证券 上海国泰 东方财富 长城证券 平安证券 南京证券 世纪证券 华福证券 广发资管 太平洋证券 联储证券 华鑫证券 东证融汇 财通证券 天风资管 第一创业 兴证证券 万联资管 中银国际 粤开证券 东吴证券 中信建投 东兴资管 华龙证券 开源证券 国海证券 中国国际 江海证券 方正资管 华兴证券 中邮证券 银河金汇 华金证券 国信证券 申港证券 东海证券 上海东方 华安证券 国联证券 上海证券 民生证券 招商证券 上海海通 信达证券 财达证券 
总共有66家公司参评


In [16]:
company_dict

{11: defaultdict(list,
             {'公司名称': '中信期货',
              '基础信息': [   公司名称                   产品全称       成立日期                 清盘日期  投资类型  \
               0  中信期货       中信期货睿丰1号集合资产管理计划 2020-12-15  2999-12-31 00:00:00   债券型   
               1  中信期货  中信期货信乐量化FOF6号集合资产管理计划 2019-11-06  2019-11-06 00:00:00  FOF型   
               
                       投资策略                                           投资策略文字描述 是否是被动指数型  \
               0  混合债券型（一级）  在深入研究的基础上积极参与可转换债券投资，以价值分析为基础，以低估值债性转债及股性转债的组合...        否   
               1    FOF/MOM  本产品主要采用量化FOF策略。子基金的投资采用多策略组合、多交易品种分散投资；子基金综合考虑...        否   
               
                 是否是QDII 是否分级 是否是海外管理人管理  产品类型 是否大集合 是否公募   预警线   止损线  业绩比较基准     备案号  
               0       否    否          否  非限定型     否    否  0.94  0.92    0.06  SNG499  
               1       否    否          否  非限定型     否    否  0.85  0.80    0.05  SGY553  ,
                  公司名称                   产品全称       成立日期                 清盘日期  投资类型  \
               0  中信期

In [34]:
basic_info = []
info_col = ['公司名称','产品全称','成立日期','清盘日期','投资类型','投资策略','投资策略文字描述','是否是被动指数型',
            '是否是QDII','是否分级','是否是海外管理人管理','产品类型','是否大集合','是否公募','预警线','止损线',
            '业绩比较基准','备案号'] \
    if yr == 2024 \
    else ['*产品全称','*成立日期','*清盘日期','*投资类型','*投资策略','投资策略文字描述','*是否是被动指数型',
            '*是否是QDII','*是否分级','*是否是海外管理人管理','*产品类型','是否大集合','是否公募','预警线','止损线',
            '业绩比较基准','*备案号']
net_value = []
nv_col = ['产品全称','日期','单位净值','累计净值','机构占比','券商自持'] if yr == 2024 else \
    ['*产品全称','*日期','*单位净值','*累计净值','*机构占比']
manager = []
mana_col = ['产品全称','开始管理日期','结束管理日期','基金经理姓名'] if yr == 2024 else \
    ['*产品全称','*开始管理日期','*结束管理日期','*基金经理姓名']
scale = []
size_col = ['产品全称','统计日期','规模(亿元)','是否为合并数据'] if yr == 2024 else \
    ['*产品全称','*统计日期','*规模(亿元)','*是否为合并数据']
split = []
split_col = ['产品全称','拆分日期','拆分比例']
bonus = []
bonus_col = ['产品全称','分红日期','每份分红金额(元)']

In [36]:
if not os.path.exists('../数据源(测试版)'):
    os.makedirs('../数据源(测试版)')

In [59]:
i = 1
all_i = len(company_dict.keys())
for num, ins_dict in company_dict.items():
    company_name = company_dict[num]['公司名称']
    print_progress(i,all_i,50,f'正在处理{company_name}..')
    i+= 1
    temp = {}
    for sheet_name, dfs in ins_dict.items():
        if sheet_name == '公司名称':
            continue
        elif sheet_name == '基础信息':
            info = pd.concat(dfs, axis=0, ignore_index=True)
            info = info.drop_duplicates().reset_index(drop=True)
            if company_name != '中邮证券':  # 对于中邮证券这个奇葩情况，真不知道怎么办
                info = info[info_col]
            temp[sheet_name] = info
        elif sheet_name == '产品净值':
            nv = pd.concat(dfs, axis=0, ignore_index=True)
            if company_name == '财达证券':
                nv.columns = nv_col
            nv_col_prb = {'产品名称':'产品全称','累计单位净值':'累计净值','净值日期':'日期'}
            for name in nv_col_prb.keys():
                if name in nv.columns:
                    nv = nv.rename(columns={name:nv_col_prb[name]})
            nv = nv.drop(nv[nv['日期'].isnull()].index)
            nv = nv.drop_duplicates().reset_index(drop=True)
            nv = nv[nv_col]
            temp[sheet_name] = nv
        elif sheet_name == '历任基金经理信息':
            mag = pd.concat(dfs, axis=0, ignore_index=True)
            mag_col_prb = {'任职起始日':'开始管理日期','任职终止日':'结束管理日期','投资主办':'基金经理姓名'}
            for name in mag_col_prb.keys():
                if name in mag.columns:
                    mag = mag.rename(columns={name:mag_col_prb[name]})
            mag = mag.drop_duplicates().reset_index(drop=True)
            mag = mag[mana_col]
            temp[sheet_name] = mag
        elif sheet_name == '规模':
            size = pd.concat(dfs, axis=0, ignore_index=True)
            size = size.drop_duplicates().reset_index(drop=True)
            size = size[size_col]
            temp[sheet_name] = size
        elif sheet_name == '拆分':
            sp = pd.concat(dfs, axis=0, ignore_index=True)
            if not sp.empty:
                sp['产品全称'] = sp['产品全称'].apply(lambda s:str(s).split(':',1)[1] if ':' in str(s) else s)
                sp = sp.drop_duplicates().reset_index(drop=True)
            sp = sp[split_col]
            temp[sheet_name] = sp
        elif sheet_name == '分红':
            bo = pd.concat(dfs, axis=0, ignore_index=True)
            if not bo.empty:
                bo['产品全称'] = bo['产品全称'].apply(lambda s:str(s).split(':',1)[1] if ':' in str(s) else s)
                bo = bo.drop_duplicates().reset_index(drop=True)
            bo = bo[bonus_col]
            temp[sheet_name] = bo
            
    with pd.ExcelWriter(f'../数据源(测试版)/{yr}证券公司集合资管计划金牛奖机构填报数据表-{company_name}.xlsx') as writer:
        for sheet_name, df in temp.items():
            df.to_excel(writer,sheet_name=sheet_name,index=False)
        
        

[##################################################] 100% | (66/66):正在处理财达证券.. 

In [ ]:
"""首创证券净值表
东证融汇经理表
中邮证券基础表(需要手动处理)
财达证券净值表"""

In [4]:
df = pd.read_excel(f'../../原始数据集/{yr}/goldenbull_zqzg/upload_data/11/2024证券公司集合资管计划金牛奖机构填报数据表.xlsx-87306454',
                   sheet_name ='基础信息')

In [5]:
info = df.copy()

In [19]:
company_name = '中信期货'

In [20]:
cols = info.columns.to_list()
cols = [elm.strip('*') for elm in cols]
info.columns = cols
info['公司名称'] = company_name

In [37]:
err = pd.DataFrame(data=None,columns=['公司名称','产品全称','投资类型','投资策略'])
prod_pairs= [('股票型','股票型'), ('混合型', '积极型混合基金'), ('混合型','偏债型混合基金'), ('混合型','平衡型混合基金'),
             ('混合型','资产配置基金'), ('债券型', '中长期纯债'), ('债券型', '短期纯债'), ('债券型', '混合债券型（一级）'),
             ('债券型', '混合债券型（二级）'), ('FOF/MOM型', 'FOF/MOM型')]

In [34]:
print(pairs)

0    (债券型, 混合债券型（一级）)
1     (FOF型, FOF/MOM)
dtype: object


In [40]:
pairs = pd.Series(zip(info['投资类型'],info['投资策略']))

for index,pair in enumerate(pairs):
    if pair not in prod_pairs:
        err = pd.concat([err,info[info.index == index][['公司名称','产品全称','投资类型','投资策略']]],axis=0,ignore_index=True)

In [41]:
err

,公司名称,产品全称,投资类型,投资策略
0,中信期货,中信期货信乐量化FOF6号集合资产管理计划,FOF型,FOF/MOM
1,中信期货,中信期货信乐量化FOF6号集合资产管理计划,FOF型,FOF/MOM


In [43]:
import pandas as pd, os, warnings, sys ,shutil
from collections import defaultdict
from datetime import datetime

warnings.filterwarnings('ignore')


def parse_dates(date_series,formats=None):
    date_formats = ['%Y-%m-%d', '%Y.%m.%d', '%Y%m%d', '%Y/%m/%d']
    # formats是列表形式，可自定义添加存在的字符串格式
    if formats is None:
        formats = []
    date_formats += formats
    for fmt in date_formats:
        # 尝试解析日期
        parsed_dates = pd.to_datetime(date_series, format=fmt, errors='coerce')
        # 合并成功解析的日期
        date_series = date_series.mask(parsed_dates.notna(), parsed_dates)
    return pd.to_datetime(date_series, errors='coerce')


def print_progress(current, total, bar_length=50,extra_text=''):
    progress = current / total
    block = int(round(bar_length * progress))
    text = f"[{'#' * block + '-' * (bar_length - block)}] {int(progress * 100)}% | ({current}/{total}):{extra_text}"

    # 使用 '\r' 返回行首，'\033[K' 清除行内剩余字符
    sys.stdout.write(f'\r{text}\033[K')
    sys.stdout.flush()


def copy_file(src, dst, tag_=0):       # 从源路径（src)复制一个文件到目标路径(path)，
    tag_update = 0
    if not os.path.exists(src):
        print(f'源文件{src}不存在！无法复制。')
    else:
        if tag_ == 1:  # 当tag_ = 1时，如果文件已存在则进行替换(多为输出的文件)，其中代码和说明文件不进行替换。(除了更新文件)
            if os.path.exists(dst):
                tag_update = 1
                os.remove(dst)
        if not os.path.exists(dst):
            shutil.copyfile(src, dst)
            if tag_ == 1:
                if tag_update == 1:
                    print(f'已更新{src}--->{dst}')
                else:
                    print(f'已复制{src}--->{dst}')
            else:
                print(f'已复制{src}--->{dst}')
        else:
            print(f'{dst}已有文件，不进行复制')

with open('../参评信息/年份信息.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()
year = lines[0]
yr = int(year)

source_dir = [f'../../原始数据集/{yr}/goldenbull_zqzg/upload_data',f'../../原始数据集/{yr}/goldenbull_zqzg/upload_data_ex']
user_info = pd.read_excel(f'../../原始数据集/{yr}/用户信息表.xlsx',skiprows=[0],dtype={'ID':int})

user_info.info()
if yr == 2024:
    num_dict = {25:'浙商证券', 29:'长江证券', 37:'国泰君安', 46:'广发资管', 47:'太平洋证券', 53:'天风证券', 55:'第一创业',
                58:'万联证券', 64:'东兴证券', 68:'中金公司', 70:'方正证券'}
else:
    num_dict = {} ##待添加

sheetname = ['基础信息', '产品净值', '历任基金经理信息', '规模', '拆分', '分红']

company_dict = {}
error_file = defaultdict(list)

for src in source_dir:
    for subdir, dirs, files in os.walk(src):
        folder_name = os.path.basename(subdir)
        if folder_name in ['1', 'null']:
            continue
        else:
            try:
                num = int(folder_name)
            except:
                continue

            if num not in company_dict.keys():
                company_dict[num] = defaultdict(list)
            if '公司名称' not in company_dict[num].keys():
                company_name_org = str(user_info.loc[user_info['ID'] == num, '公司名称'].values[0])
                # if isinstance(company_name_org, str) and company_name_org != 'nan':
                #     company_name = company_name_org[:4] if '太平洋' not in company_name_org else company_name_org[:5]
                # else:
                #     company_name = num_dict[num]
                if num in num_dict.keys():
                    company_name = num_dict[num]
                else:
                    company_name = company_name_org[:4]
            else:
                company_name = company_dict[num]['公司名称']
            print('\n\n')
            print(f'{num} {company_name}'.center(80, '-'))
            i = 1
            er_file = []
        for file in files:
            # 检查文件名中是否包含'.xls'
            if '.xls' in file.lower():
                file_path = os.path.join(subdir, file)
                xls = pd.ExcelFile(file_path)
                sn = xls.sheet_names
                if len(sn) < 4:
                    if len(sn) == 1:
                        if '净值' in file: sheet_name = '产品净值'
                        elif '经理' in file: sheet_name = '历任基金经理信息'
                        elif '规模' in file: sheet_name = '规模'
                        elif '基础' in file: sheet_name = '基础信息'
                        elif '分红' in file: sheet_name = '分红'
                        elif '拆分' in file: sheet_name = '拆分'
                        df = pd.read_excel(xls)
                        cols = df.columns.to_list()
                        col = [elm.strip('*').strip() for elm in cols]
                        df.columns = col
                        company_dict[num][sheet_name].append(df)
                    else:
                        error_file[num].append(file)
                        er_file.append(file)
                else:
                    for sheet_name in sn:
                        df = pd.read_excel(xls, sheet_name=sheet_name)

                        # 规范化
                        if '基础' in sheet_name: sheet_name = '基础信息'
                        elif '净值' in sheet_name: sheet_name = '产品净值'
                        elif '经理' in sheet_name: sheet_name = '历任基金经理信息'
                        elif '分红' in sheet_name: sheet_name = '分红'
                        elif '规模' in sheet_name: sheet_name = '规模'
                        elif '拆分' in sheet_name: sheet_name = '拆分'

                        if sheet_name == '基础信息':
                            cols = df.columns.to_list()
                            if '公司名称' not in company_dict[num].keys():
                                # comfirm = input(f'\n输入规定的{num}公司名称,现<<{company_name}>>原<<{company_name_org}>>\n(若已可行，回车即可)：')
                                """confirm = ''
                                ## 注：新一年的评奖时把封印解开！
                                if not comfirm:
                                    pass
                                else:
                                    company_name = comfirm#"""
                                company_dict[num]['公司名称'] = company_name
                            else:
                                company_name = company_dict[num]['公司名称']
                            df['公司名称'] = company_name
                            df = df[['公司名称'] + cols]

                        cols = df.columns.to_list()
                        col = [elm.strip('*').strip() for elm in cols]
                        df.columns = col
                        company_dict[num][sheet_name].append(df)

                    if i + 1 <= len(files):
                        i = i + 1
                    print_progress(i, len(files), 10, f'正在读取:{file}')
                if '分红' not in company_dict[num].keys():
                    print(f'\n{file}少了分红')
                    company_dict[num]['分红'].append(
                        pd.DataFrame(data=None, columns=['产品全称', '分红日期', '每份分红金额(元)']))
                if '拆分' not in company_dict[num].keys():
                    print(f'\n{file}少了拆分')
                    company_dict[num]['拆分'].append(
                        pd.DataFrame(data=None, columns=['产品全称', '拆分日期', '拆分比例']))
            else:
                error_file[num].append(file)
                er_file.append(file)
        if len(er_file) > 0:
            print(f"\n出错文件：{','.join(er_file)}")

for i in company_dict.keys():
    print(company_dict[i]['公司名称'],end=' ')
number = len(company_dict.keys())
print('\n')
print(f'总共有{number}家公司参评'.center(80,'-'))
print()

info_col = ['公司名称','产品全称','成立日期','清盘日期','投资类型','投资策略','投资策略文字描述','是否是被动指数型',
            '是否是QDII','是否分级','是否是海外管理人管理','产品类型','是否大集合','是否公募','预警线','止损线',
            '业绩比较基准','备案号'] \
    if yr == 2024 \
    else ['产品全称','成立日期','清盘日期','投资类型','投资策略','投资策略文字描述','是否是被动指数型',
            '是否是QDII','是否分级','是否是海外管理人管理','产品类型','是否大集合','是否公募','预警线','止损线',
            '业绩比较基准','备案号']
nv_col = ['产品全称','日期','单位净值','累计净值','机构占比','券商自持'] if yr == 2024 else \
    ['产品全称','日期','单位净值','累计净值','机构占比']
mana_col = ['产品全称','开始管理日期','结束管理日期','基金经理姓名']
size_col = ['产品全称','统计日期','规模(亿元)','是否为合并数据']
split_col = ['产品全称','拆分日期','拆分比例']
bonus_col = ['产品全称','分红日期','每份分红金额(元)']



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      88 non-null     int32 
 1   用户名     88 non-null     object
 2   电话      88 non-null     object
 3   邮箱      88 non-null     object
 4   公司代码    88 non-null     object
 5   公司地址    35 non-null     object
 6   公司名称    60 non-null     object
 7   公司备案    13 non-null     object
 8   注册地址    35 non-null     object
 9   办公地址    60 non-null     object
 10  注册时间    60 non-null     object
 11  公司简介    37 non-null     object
dtypes: int32(1), object(11)
memory usage: 8.0+ KB



------------------------------------11 中信期货-------------------------------------
[##########] 100% | (4/4):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表.xlsx-87306454
出错文件：券商集合资产管理人评选指标补充.docx-02426627



------------------------------------13 华宝证券-------------------------------------
[##########] 100% | (3/3):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-华宝证券资管.xlsx-5

[##########] 100% | (1/1):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-华龙证券.xlsx-48942215


------------------------------------66 开源证券-------------------------------------
[##########] 100% | (3/3):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表权益.xlsx-84653041K


------------------------------------67 国海证券-------------------------------------

【产品规模】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-77829701少了分红

【产品规模】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-77829701少了拆分
[####------] 41% | (5/12):正在读取:【投资策略已更新】【国海证券】2024证券公司集合资产管理计划金牛奖机构填报数据表.xlsx-10946141
出错文件：【产品规模】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-77829701,【净值、机构占比等】2024证券公司集合资产管理计划金牛奖机构填报数据表(1).xlsx-57615308,【分红记录】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-13701780,【基金经理信息】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-17006055,【拆分记录】2024证券公司集合资产管理计划金牛奖机构填报数据表（国海证券）.xlsx-63102060



------------------------------------68 中金公司-------------------------------------
[##########] 100% | (3/3):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-中金公司.xlsx-65154672


------------------------------------69 江海证券------

[##########] 100% | (1/1):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-华鑫证券-修正.xlsx-09805245


------------------------------------52 财通证券-------------------------------------

出错文件：固收财通证券资管-2024资管金牛奖填报.pdf-35973712,权益财通证券资管-2024资管金牛奖填报.pdf-61508239,财通证券资管2024资管金牛申报材料.docx-22223143,财通证券资管2024资管金牛申报材料.pdf-10237595



------------------------------------53 天风证券-------------------------------------

出错文件：盖章附件（用印版）.pdf-96087035



------------------------------------57 兴证证券-------------------------------------

出错文件：2024证券公司集合资管计划金牛奖机构填报数据表.pdf-02699646



------------------------------------58 万联证券-------------------------------------

出错文件：万联添利粤升1号.png-82685808,万联聚享添利1号.png-43189603,万联聚享添利3号.png-72090783,万联聚享添利5号.png-53763315,万联证券添利粤信3号.png-34756769,万联证券资产管理（广东）有限公司【万联资管】集合产品WIND同类产品排名2023年12月末截图证明.docx-68411032,万联证券资产管理（广东）有限公司【万联资管】集合产品WIND同类产品排名2023年12月末截图证明——PART1.docx-52022104,万联证券资产管理（广东）有限公司【万联资管】集合产品WIND同类产品排名2023年12月末截图证明——PART2.docx-49746312,万联财星东照月添利1号.png-12624428,产品基本信息——万联资管.pdf-03890073,

In [45]:
if not os.path.exists('../数据源(测试版)'):
    os.makedirs('../数据源(测试版)')
else:
    shutil.rmtree('../数据源(测试版)')
    os.makedirs('../数据源(测试版)')

In [88]:
def find_name_err(df):
    new = name_err.copy()
    df_ex = df.drop_duplicates(subset=['产品全称']).reset_index(drop=True).copy()
    df_ex['表名'] = sheet_name
    df_ex['公司名称'] = company_name
    name_df = df_ex['产品全称']
    for index, name in enumerate(name_df):
        if len(name) >= 35 or any(c in name for c in ['(','（',':','：',')','）','\n']): 
            new = pd.concat([new, df_ex[df_ex.index == index][['公司名称','表名','产品全称']]],
                                axis=0, ignore_index=True)
    return new

In [89]:
cate_error = defaultdict(list)

err = pd.DataFrame(data=None,columns=['公司名称','产品全称','投资类型','投资策略'])
prod_pairs= [('股票型','股票型'), ('混合型', '积极型混合基金'), ('混合型','偏债型混合基金'), ('混合型','平衡型混合基金'),
             ('混合型','资产配置基金'), ('债券型', '中长期纯债'), ('债券型', '短期纯债'), ('债券型', '混合债券型（一级）'),
             ('债券型', '混合债券型（二级）'), ('FOF/MOM型', 'FOF/MOM型')]

name_err = pd.DataFrame(data=None,columns=['公司名称','表名','产品全称'])
i = 1
all_i = len(company_dict.keys())
for num, ins_dict in company_dict.items():
    company_name = company_dict[num]['公司名称']
    print_progress(i, all_i, 50, f'正在处理{company_name}..')
    i += 1
    temp = {}
    for sheet_name, dfs in ins_dict.items():
        if sheet_name == '公司名称':
            continue
        elif sheet_name == '基础信息':
            info = pd.concat(dfs, axis=0, ignore_index=True)
            if company_name != '中邮证券':
                info = info[info_col]
            else:
                info.rename(columns={'':'产品全称'},inplace=True)
                # """ 2024年特例
            info = info.drop_duplicates(subset='产品全称', keep='last').reset_index(drop=True).dropna(how='all')
            info['成立日期'] = parse_dates(info['成立日期']).dt.strftime('%Y/%m/%d')
            info['清盘日期'] = info['清盘日期'].fillna('2999/12/31')
            
            info['产品全称'] = info['产品全称'].apply(lambda x:str(x).strip())
            
            pairs = pd.Series(zip(info['投资类型'], info['投资策略']))

            for index, pair in enumerate(pairs):
                if pair not in prod_pairs:
                    err = pd.concat([err, info[info.index == index][['公司名称', '产品全称', '投资类型', '投资策略']]],
                                    axis=0, ignore_index=True)
            name_err = find_name_err(info)
                    
            temp[sheet_name] = info
        elif sheet_name == '产品净值':
            nv = pd.concat(dfs, axis=0, ignore_index=True)
            if company_name == '财达证券':
                nv.columns = nv_col
            nv_col_prb = {'产品名称': '产品全称', '累计单位净值': '累计净值', '净值日期': '日期'}
            for name in nv_col_prb.keys():
                if name in nv.columns:
                    nv = nv.rename(columns={name: nv_col_prb[name]})
            nv = nv.drop(nv[nv['日期'].isnull()].index)
            nv['产品全称'] = nv['产品全称'].apply(lambda x:str(x).strip())
            nv = nv[nv_col]
            nv = nv.drop_duplicates(subset=['产品全称', '日期']).reset_index(drop=True).dropna(how='all')
            
            name_err = find_name_err(nv)
            
            temp[sheet_name] = nv
        elif sheet_name == '历任基金经理信息':
            mag = pd.concat(dfs, axis=0, ignore_index=True)
            mag_col_prb = {'任职起始日': '开始管理日期', '任职终止日': '结束管理日期', '投资主办': '基金经理姓名'}
            for name in mag_col_prb.keys():
                if name in mag.columns:
                    mag = mag.rename(columns={name: mag_col_prb[name]})
            mag['产品全称'] = mag['产品全称'].apply(lambda x:str(x).strip())
            mag = mag.drop_duplicates().reset_index(drop=True).dropna(how='all')
            mag['开始管理日期'] = parse_dates(mag['开始管理日期']).dt.strftime('%Y/%m/%d')
            #mag['结束管理日期'] = parse_dates(mag['结束管理日期']).dt.strftime('%Y/%m/%d')
            mag = mag[mana_col]
            
            name_err = find_name_err(mag)
            
            temp[sheet_name] = mag
        elif sheet_name == '规模':
            size = pd.concat(dfs, axis=0, ignore_index=True)
            size = size.drop_duplicates(subset=['产品全称','统计日期']).reset_index(drop=True).dropna(how='all')
            size = size[size_col]
            size['产品全称'] = size['产品全称'].apply(lambda x:str(x).strip())
            size['统计日期'] = parse_dates(size['统计日期']).dt.strftime('%Y/%m/%d')
            
            name_err = find_name_err(size)
            
            temp[sheet_name] = size
        elif sheet_name == '拆分':
            sp = pd.concat(dfs, axis=0, ignore_index=True)
            if not sp.empty:
                sp['产品全称'] = sp['产品全称'].apply(lambda s: str(s).split(':', 1)[1] if ':' in str(s) else s)
                try:
                    sp['拆分比例'] = sp['拆分比例'].apply(lambda s: str(s).split(':', 1)[1] if ':' in str(s) else s)
                except: pass
                sp = sp.drop_duplicates().reset_index(drop=True).dropna(how='all')
                sp = sp.drop(sp[sp['拆分比例']=='无'].index)
                sp['产品全称'] = sp['产品全称'].apply(lambda x:str(x).strip())
            sp = sp[split_col]
            sp['拆分日期'] = parse_dates(sp['拆分日期']).dt.strftime('%Y/%m/%d')
            
            name_err = find_name_err(sp)
            
            temp[sheet_name] = sp
        elif sheet_name == '分红':
            bo = pd.concat(dfs, axis=0, ignore_index=True)
            if not bo.empty:
                bo['产品全称'] = bo['产品全称'].apply(lambda s: str(s).split(':', 1)[1] if ':' in str(s) else s)
                bo = bo.drop_duplicates().reset_index(drop=True).dropna(how='all')
                bo['产品全称'] = bo['产品全称'].apply(lambda x:str(x).strip())
            bo = bo[bonus_col]
            bo['分红日期'] = parse_dates(bo['分红日期']).dt.strftime('%Y/%m/%d')
            
            name_err = find_name_err(bo)
            
            temp[sheet_name] = bo

    with pd.ExcelWriter(
            f'../数据源(测试版)/{yr}证券公司集合资管计划金牛奖机构填报数据表-{company_name}.xlsx') as writer:
        for sheet_name, df in temp.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(name_err)
maximum = 0
for lst in error_file.values():
    if maximum < len(lst):
        maximum = len(lst)

for lst in error_file.values():
    for i in range(maximum - len(lst)):
        lst.append('')

error = pd.DataFrame(error_file)

print('')
copy_file('../资料/代码/2_合并数据表.py','../step_2 合并数据表/1_合并数据表.py')
copy_file('../资料/说明/数据源说明.docx','../step_2 合并数据表/0_数据源说明.docx')
print('')

[#-------------------------------------------------] 1% | (1/66):正在处理中信期货..Empty DataFrame
Columns: [公司名称, 表名, 产品全称]
Index: []
[##------------------------------------------------] 3% | (2/66):正在处理华宝证券..Empty DataFrame
Columns: [公司名称, 表名, 产品全称]
Index: []
[##------------------------------------------------] 4% | (3/66):正在处理华创证券..Empty DataFrame
Columns: [公司名称, 表名, 产品全称]
Index: []
[###-----------------------------------------------] 6% | (4/66):正在处理国金证券..Empty DataFrame
Columns: [公司名称, 表名, 产品全称]
Index: []
[####----------------------------------------------] 7% | (5/66):正在处理安信证券..   公司名称    表名                           产品全称
0  安信证券  产品净值  安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
[#####---------------------------------------------] 9% | (6/66):正在处理东莞证券..   公司名称    表名                           产品全称
0  安信证券  产品净值  安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
[#####---------------------------------------------] 10% | (7/66):正在处理中航证券..   公司名称    表名                           产品全称
0  安信证券  产品净值  安信资管瑞元添利一年持有（安信理财1号，参公大集合转

[##############------------------------------------] 27% | (18/66):正在处理首创证券..    公司名称        表名                                产品全称
0   安信证券      产品净值       安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
2   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
3   浙商证券      基础信息        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
4   浙商证券      产品净值        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
5   浙商证券      产品净值        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
6   浙商证券      产品净值        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
7   浙商证券  历任基金经理信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
8   浙商证券  历任基金经理信息        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
9   浙商证券  历任基金经理信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
10  浙商证券        规模          浙商汇金卓越配置一年持有期混合型基金中基金（FOF）
11  浙商证券        规模        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
12  浙商证券        规模        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
13  浙商证券        规模        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
14  中信证券      基础信息  中信证券财富优选一年持有期混合型基金中基金（FOF）集合资产管理计划
15  中信证券      产品净值  中信证券财富优选一年持有期混合型基金中基金（F

[#################---------------------------------] 33% | (22/66):正在处理华泰证券..    公司名称        表名                                产品全称
0   安信证券      产品净值       安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
2   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
3   浙商证券      基础信息        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
4   浙商证券      产品净值        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
5   浙商证券      产品净值        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
6   浙商证券      产品净值        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
7   浙商证券  历任基金经理信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
8   浙商证券  历任基金经理信息        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
9   浙商证券  历任基金经理信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
10  浙商证券        规模          浙商汇金卓越配置一年持有期混合型基金中基金（FOF）
11  浙商证券        规模        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
12  浙商证券        规模        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
13  浙商证券        规模        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
14  中信证券      基础信息  中信证券财富优选一年持有期混合型基金中基金（FOF）集合资产管理计划
15  中信证券      产品净值  中信证券财富优选一年持有期混合型基金中基金（F

[###################-------------------------------] 37% | (25/66):正在处理长城证券..    公司名称        表名                                产品全称
0   安信证券      产品净值       安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
2   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
3   浙商证券      基础信息        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
4   浙商证券      产品净值        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
5   浙商证券      产品净值        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
6   浙商证券      产品净值        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
7   浙商证券  历任基金经理信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
8   浙商证券  历任基金经理信息        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
9   浙商证券  历任基金经理信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
10  浙商证券        规模          浙商汇金卓越配置一年持有期混合型基金中基金（FOF）
11  浙商证券        规模        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
12  浙商证券        规模        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
13  浙商证券        规模        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
14  中信证券      基础信息  中信证券财富优选一年持有期混合型基金中基金（FOF）集合资产管理计划
15  中信证券      产品净值  中信证券财富优选一年持有期混合型基金中基金（F

[#####################-----------------------------] 42% | (28/66):正在处理世纪证券..    公司名称        表名                                产品全称
0   安信证券      产品净值       安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
2   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
3   浙商证券      基础信息        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
4   浙商证券      产品净值        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
5   浙商证券      产品净值        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
6   浙商证券      产品净值        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
7   浙商证券  历任基金经理信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
8   浙商证券  历任基金经理信息        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
9   浙商证券  历任基金经理信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
10  浙商证券        规模          浙商汇金卓越配置一年持有期混合型基金中基金（FOF）
11  浙商证券        规模        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
12  浙商证券        规模        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
13  浙商证券        规模        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
14  中信证券      基础信息  中信证券财富优选一年持有期混合型基金中基金（FOF）集合资产管理计划
15  中信证券      产品净值  中信证券财富优选一年持有期混合型基金中基金（F

[#########################-------------------------] 50% | (33/66):正在处理华鑫证券..    公司名称        表名                                 产品全称
0   安信证券      产品净值        安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息         浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
2   浙商证券      基础信息         浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
3   浙商证券      基础信息         浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
4   浙商证券      产品净值         浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
..   ...       ...                                  ...
59  广发资管      产品净值  广发资管智荟广易六个月持有期混合型基金中基金（FOF）集合资产管理计划
60  广发资管  历任基金经理信息       广发资管全球精选一年持有期债券型集合资产管理计划（QDII）
61  广发资管  历任基金经理信息  广发资管智荟广易六个月持有期混合型基金中基金（FOF）集合资产管理计划
62  广发资管        规模       广发资管全球精选一年持有期债券型集合资产管理计划（QDII）
63  广发资管        规模  广发资管智荟广易六个月持有期混合型基金中基金（FOF）集合资产管理计划

[64 rows x 3 columns]
[##########################------------------------] 51% | (34/66):正在处理东证融汇..    公司名称        表名                                 产品全称
0   安信证券      产品净值        安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息         浙商汇金卓越配置一年持有

[#################################-----------------] 66% | (44/66):正在处理东兴证券..    公司名称        表名                                产品全称
0   安信证券      产品净值       安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
2   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
3   浙商证券      基础信息        浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
4   浙商证券      产品净值        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
..   ...       ...                                 ...
74  兴证证券      基础信息  兴证资管金麒麟3个月持有期混合型基金中基金(FOF)集合资产管理计划
75  兴证证券      产品净值  兴证资管金麒麟3个月持有期混合型基金中基金(FOF)集合资产管理计划
76  兴证证券  历任基金经理信息  兴证资管金麒麟3个月持有期混合型基金中基金(FOF)集合资产管理计划
77  兴证证券        规模  兴证资管金麒麟3个月持有期混合型基金中基金(FOF)集合资产管理计划
78  兴证证券        分红  兴证资管金麒麟3个月持有期混合型基金中基金(FOF)集合资产管理计划

[79 rows x 3 columns]
[##################################----------------] 68% | (45/66):正在处理华龙证券..    公司名称        表名                                产品全称
0   安信证券      产品净值       安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息        浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B

[##########################################--------] 83% | (55/66):正在处理国信证券..    公司名称        表名                               产品全称
0   安信证券      产品净值      安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息       浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
2   浙商证券      基础信息       浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
3   浙商证券      基础信息       浙商汇金鼎盈事件驱动灵活配置混合型证券投资基金（LOF）
4   浙商证券      产品净值       浙商汇金卓越配置一年持有期混合型基金中基金（FOF）A类
..   ...       ...                                ...
84  国信证券      基础信息  国信经典组合三个月持有期混合型基金中基金(FOF)集合资产管理计划
85  国信证券      产品净值  国信经典组合三个月持有期混合型基金中基金(FOF)集合资产管理计划
86  国信证券  历任基金经理信息  国信经典组合三个月持有期混合型基金中基金(FOF)集合资产管理计划
87  国信证券        规模  国信经典组合三个月持有期混合型基金中基金(FOF)集合资产管理计划
88  国信证券        拆分  国信经典组合三个月持有期混合型基金中基金(FOF)集合资产管理计划

[89 rows x 3 columns]
[##########################################--------] 84% | (56/66):正在处理申港证券..    公司名称        表名                               产品全称
0   安信证券      产品净值      安信资管瑞元添利一年持有（安信理财1号，参公大集合转型前）
1   浙商证券      基础信息       浙商汇金卓越配置一年持有期混合型基金中基金（FOF）B类
2   浙商证券     

In [91]:
with pd.ExcelWriter('../输出文件/其他/错误.xlsx') as writer:
    error.to_excel(writer, sheet_name = '文件读取错误', index=False)
    err.to_excel(writer, sheet_name='策略错误', index=False)
    name_err.to_excel(writer, sheet_name = '名称错误', index=False)